# Cenotoo REST API — Soil Sensor Demo (plain HTTP)

End-to-end walkthrough of the **current** Cenotoo REST API using only `requests`:
authentication, projects, API keys, collections, data ingestion, filtered queries,
time-series statistics, deletion, and cleanup.

| Concern | How it is handled |
|---|---|
| Management (projects, keys, collections) | JWT bearer token from `POST /api/v1/token` |
| Ingestion | project **write** API key via `X-API-Key` |
| Queries / statistics | project **read** API key via `X-API-Key` |
| Deletion | project **master** API key via `X-API-Key` |
| Base URL | `CENOTOO_BASE_URL` (no `/api/v1` suffix) |

Every step prints the request outcome, and the final cells delete exactly what they created.


## 1. Install dependencies

In [ ]:
%pip install -q requests python-dotenv

## 2. Configuration

Set these via environment variables (or edit the defaults):

| Variable | Meaning | Default |
|---|---|---|
| `CENOTOO_BASE_URL` | API origin, **without** `/api/v1` | `http://localhost:8000` |
| `CENOTOO_USERNAME` | admin/user login | `nostradamus` |
| `CENOTOO_PASSWORD` | user password | *(empty — required)* |
| `CENOTOO_CA_BUNDLE` | PEM CA bundle for private/self-signed TLS | *(unset → system trust)* |
| `CENOTOO_VERIFY_TLS` | set `false` to skip TLS verification (dev only) | `true` |

> The organization is **not** passed in URLs; it is derived from the authenticated user / API key.


In [ ]:
import json
import os
import random
import re
import time
from datetime import datetime, timedelta, timezone

import requests
from dotenv import load_dotenv

load_dotenv()

BASE_URL = os.getenv("CENOTOO_BASE_URL", "http://localhost:8000").rstrip("/")
API = f"{BASE_URL}/api/v1"

USERNAME = os.getenv("CENOTOO_USERNAME", "nostradamus")
PASSWORD = os.getenv("CENOTOO_PASSWORD", "")

CA_BUNDLE = os.getenv("CENOTOO_CA_BUNDLE", "")
VERIFY = CA_BUNDLE if CA_BUNDLE else (
    os.getenv("CENOTOO_VERIFY_TLS", "true").lower() not in ("false", "0", "no")
)

assert PASSWORD, "Set CENOTOO_PASSWORD (and optionally CENOTOO_BASE_URL) before running."

def unique_name(prefix):
    stamp = time.strftime("%Y%m%d%H%M%S")
    return f"{prefix}_{stamp}_{random.randint(1000, 9999)}"

print(f"Target API: {API}")
print(f"TLS verify: {VERIFY!r}")

## 3. Health check and authentication

`POST /api/v1/token` uses the OAuth2 **password** flow (form-encoded, not JSON).
The response contains `access_token`, `refresh_token`, and `token_type`.

In [ ]:
health = requests.get(f"{BASE_URL}/health", timeout=10, verify=VERIFY)
print("health:", health.status_code, health.json())
health.raise_for_status()

token_resp = requests.post(
    f"{API}/token",
    data={"username": USERNAME, "password": PASSWORD},
    timeout=15,
    verify=VERIFY,
)
print("token:", token_resp.status_code, token_resp.json().get("token_type"))
token_resp.raise_for_status()

ACCESS_TOKEN = token_resp.json()["access_token"]

# Session for management calls (JWT bearer).
SESSION = requests.Session()
SESSION.verify = VERIFY
SESSION.headers.update({"Authorization": f"Bearer {ACCESS_TOKEN}"})

# Session for project-scoped API-key calls (no bearer header, to prove key auth).
KEY_SESSION = requests.Session()
KEY_SESSION.verify = VERIFY


def api_call(method, path, **kwargs):
    """Call the API with the JWT bearer session."""
    resp = SESSION.request(method, f"{API}{path}", timeout=120, **kwargs)
    if resp.status_code >= 400:
        print(f"{method} {path} -> {resp.status_code}: {resp.text}")
    resp.raise_for_status()
    return resp


def key_call(method, path, api_key, **kwargs):
    """Call the API with a scoped project API key (X-API-Key header)."""
    headers = dict(kwargs.pop("headers", {}))
    headers["X-API-Key"] = api_key
    resp = KEY_SESSION.request(method, f"{API}{path}", headers=headers, timeout=120, **kwargs)
    if resp.status_code >= 400:
        print(f"{method} {path} -> {resp.status_code}: {resp.text}")
    resp.raise_for_status()
    return resp


def show(label, value):
    print(f"\n{label}:")
    print(json.dumps(value, indent=2, default=str)[:4000])

## 4. Create a project

`POST /api/v1/projects` returns `{"message": ..., "id": {"project_id": ...}}`.
Project and collection names may contain only latin letters, numbers, and underscores.

In [ ]:
project_name = unique_name("soil_project")
project_resp = api_call(
    "POST",
    "/projects",
    json={
        "project_name": project_name,
        "description": "Soil monitoring demo (plain HTTP notebook)",
        "tags": ["soil", "agriculture", "notebook"],
    },
).json()
show("create project", project_resp)

PROJECT_ID = project_resp["id"]["project_id"]
print("PROJECT_ID =", PROJECT_ID)

## 5. Create project API keys

`POST /api/v1/projects/{project_id}/keys` with `{"key_type": "read"|"write"|"master"}`.
The **raw secret is returned only once** at creation; listing keys later shows hashes.

In [ ]:
def create_project_key(project_id, key_type):
    data = api_call(
        "POST", f"/projects/{project_id}/keys", json={"key_type": key_type}
    ).json()
    print(f"created {key_type} key: {data['api_key'][:6]}…")
    return data["api_key"]


MASTER_KEY = create_project_key(PROJECT_ID, "master")
WRITE_KEY = create_project_key(PROJECT_ID, "write")
READ_KEY = create_project_key(PROJECT_ID, "read")

## 6. Create a collection

`collection_schema` is an **example record**: the API infers column types from the Python
values (`str → text`, `int`, `float → decimal`, `bool`) and always adds `timestamp`, `day`,
and `id`. `key` is auto-added as `TEXT` if omitted.

The create response embeds the new UUID in its `message`; the list endpoint is the
structured fallback used here.

In [ ]:
collection_name = unique_name("soil_sensors")

collection_schema = {
    "key": "SOIL_001",                 # text
    "timestamp": "2025-06-17T10:30:00Z",
    "soil_moisture": 45.2,             # float
    "ph_level": 6.8,                   # float
    "temperature": 18.5,               # float
    "nitrogen": 45.0,                  # float
    "phosphorus": 12.5,                # float
    "potassium": 180.0,                # float
    "battery_level": 87.3,             # float
}

create_resp = key_call(
    "POST",
    f"/projects/{PROJECT_ID}/collections",
    MASTER_KEY,
    json={
        "name": collection_name,
        "description": "Soil monitoring data",
        "tags": ["soil", "agriculture", "sensors"],
        "collection_schema": collection_schema,
    },
).json()
show("create collection", create_resp)

match = re.search(r"ID\s+([0-9a-fA-F-]{36})", create_resp.get("message", ""))
COLLECTION_ID = match.group(1) if match else None

if not COLLECTION_ID:
    items = api_call("GET", f"/projects/{PROJECT_ID}/collections").json()["items"]
    COLLECTION_ID = next(
        c["collection_id"] for c in items if c["collection_name"] == collection_name
    )

print("COLLECTION_ID =", COLLECTION_ID)
show("collection info", api_call(
    "GET", f"/projects/{PROJECT_ID}/collections/{COLLECTION_ID}"
).json())

## 7. Generate soil sensor data (3 sensors × 24 hourly readings)

In [ ]:
def generate_soil_data(num_sensors=3, num_readings=24):
    """Generate soil sensor readings over the last 24 hours."""
    data = []
    base_time = datetime.now(timezone.utc).replace(microsecond=0) - timedelta(hours=num_readings)

    for sensor_num in range(1, num_sensors + 1):
        sensor_id = f"SOIL_{sensor_num:03d}"
        for hour in range(num_readings):
            timestamp = base_time + timedelta(hours=hour)
            data.append({
                "key": sensor_id,
                "timestamp": timestamp.strftime("%Y-%m-%dT%H:%M:%SZ"),
                "soil_moisture": round(random.uniform(30, 65), 1),
                "ph_level": round(random.uniform(6.0, 7.5), 1),
                "temperature": round(random.uniform(15, 25), 1),
                "nitrogen": round(random.uniform(30, 60), 1),
                "phosphorus": round(random.uniform(8, 20), 1),
                "potassium": round(random.uniform(120, 220), 1),
                "battery_level": round(random.uniform(70, 100), 1),
            })
    return data


soil_data = generate_soil_data(num_sensors=3, num_readings=24)
soil_data[0]

## 8. Send data (write API key)

`POST /api/v1/projects/{project_id}/collections/{collection_id}/send_data` accepts a
single object or a list. Each record is validated against the collection schema; the
response reports `processed_count`. Data flows through Kafka, so Cassandra visibility is
eventually consistent.

In [ ]:
send_resp = key_call(
    "POST",
    f"/projects/{PROJECT_ID}/collections/{COLLECTION_ID}/send_data",
    WRITE_KEY,
    json=soil_data,
).json()
show("send_data", send_resp)
assert send_resp["processed_count"] == len(soil_data)

## 9. Wait for the Kafka → Cassandra write path

Writes are asynchronous. Poll `get_data` (without `limit`, so `total_count` is the real
count) until all records are visible.

In [ ]:
def wait_for_total(project_id, collection_id, read_key, want, timeout=180, interval=3):
    deadline = time.monotonic() + timeout
    last = -1
    while time.monotonic() < deadline:
        body = key_call(
            "GET",
            f"/projects/{project_id}/collections/{collection_id}/get_data",
            read_key,
        ).json()
        last = body.get("total_count", 0)
        if last >= want:
            return last
        time.sleep(interval)
    raise TimeoutError(f"only {last}/{want} records visible after {timeout}s")


visible = wait_for_total(PROJECT_ID, COLLECTION_ID, READ_KEY, len(soil_data))
print(f"records visible: {visible}")

## 10. Query data

`GET .../get_data` query parameters:

| Param | Encoding |
|---|---|
| `attributes` | repeated query parameter (`?attributes=key&attributes=temperature`) |
| `filters` | JSON-encoded list of `{property_name, operator, property_value}` |
| `order_by` | JSON-encoded `{"field": ..., "order": "asc"|"desc"}` |
| `limit`, `offset`, `nested` | integers / boolean |

Response: `{"data": [...], "total_count": N}`.
Note: `total_count` counts the rows fetched (capped by `limit`) — treat it as exact only
when `limit` is omitted.

In [ ]:
base = f"/projects/{PROJECT_ID}/collections/{COLLECTION_ID}/get_data"

all_data = key_call("GET", base, READ_KEY).json()
print("total_count:", all_data["total_count"], "| returned:", len(all_data["data"]))

limited = key_call("GET", base, READ_KEY, params={"limit": 5}).json()
show("limit=5", limited)

ordered = key_call(
    "GET",
    base,
    READ_KEY,
    params={
        "attributes": ["key", "temperature"],
        "order_by": json.dumps({"field": "temperature", "order": "desc"}),
        "limit": 5,
    },
).json()
show("top 5 temperatures (attributes + order_by)", ordered)

## 11. Filters: equality, comparison, AND, OR

In [ ]:
# Single condition: one sensor
sensor_filter = [{"property_name": "key", "operator": "eq", "property_value": "SOIL_001"}]
so = key_call("GET", base, READ_KEY, params={"filters": json.dumps(sensor_filter), "limit": 5}).json()
show("key == SOIL_001 (limit 5)", so)

# Numeric comparison: moisture above 55%
moisture_filter = [{"property_name": "soil_moisture", "operator": "gt", "property_value": 55}]
show("soil_moisture > 55", key_call(
    "GET", base, READ_KEY, params={"filters": json.dumps(moisture_filter), "limit": 5}
).json())

# Multiple conditions are AND-ed
alert_filter = [
    {"property_name": "soil_moisture", "operator": "lt", "property_value": 35},
    {"property_name": "battery_level", "operator": "lt", "property_value": 80},
]
show("soil_moisture < 35 AND battery_level < 80", key_call(
    "GET", base, READ_KEY, params={"filters": json.dumps(alert_filter), "limit": 5}
).json())

# OR uses an operator object with nested `operands`
or_filter = [{
    "operator": "or",
    "operands": [
        {"property_name": "key", "operator": "eq", "property_value": "SOIL_001"},
        {"property_name": "key", "operator": "eq", "property_value": "SOIL_002"},
    ],
}]
show("key == SOIL_001 OR key == SOIL_002", key_call(
    "GET", base, READ_KEY, params={"filters": json.dumps(or_filter), "limit": 5}
).json())

## 12. Time-series statistics

`GET .../statistics` query parameters: `attribute`, `stat`
(`avg|max|min|sum|count|distinct|p50|p90|p95|p99`), optional `interval`
(`every_N_minutes|hours|days|weeks|months`), `start_time`, `end_time`, `filters`,
`order`, `group_by`, `limit`.

Non-`distinct` stats return a **list** of buckets (or raw rows without `interval`).
`distinct` returns an object with per-key statistics.

In [ ]:
stats_url = f"/projects/{PROJECT_ID}/collections/{COLLECTION_ID}/statistics"

def get_statistics(api_key, **params):
    return key_call("GET", stats_url, api_key, params=params).json()


# Average moisture in 6-hour buckets (grouped per key by default)
avg_moisture = get_statistics(READ_KEY, attribute="soil_moisture", stat="avg", interval="every_6_hours")
show("avg(soil_moisture) every 6h", avg_moisture)

# Top 5 highest temperatures
show("max(temperature) desc", get_statistics(
    READ_KEY, attribute="temperature", stat="max", order="desc", limit=5
))

# Minimum battery for one sensor
show("min(battery_level) for SOIL_001", get_statistics(
    READ_KEY,
    attribute="battery_level",
    stat="min",
    filters=json.dumps(sensor_filter),
))

# Distinct pH values per sensor
show("distinct(ph_level)", get_statistics(
    READ_KEY, attribute="ph_level", stat="distinct", limit=20
))

## 13. Delete data (master API key)

`DELETE .../delete_data` body: optional `key`, `timestamp_from`, `timestamp_to`
(at least one required). This example deletes **all** data for `SOIL_001`.

In [ ]:
deleted = key_call(
    "DELETE",
    f"/projects/{PROJECT_ID}/collections/{COLLECTION_ID}/delete_data",
    MASTER_KEY,
    json={"key": "SOIL_001"},
).json()
show("delete_data", deleted)

gone = key_call(
    "GET", base, READ_KEY,
    params={"filters": json.dumps(sensor_filter), "limit": 5},
).json()
show("SOIL_001 after delete", gone)
assert len(gone["data"]) == 0, "expected SOIL_001 records to be gone"

## 14. Cleanup

Delete the collection, then all API keys, then the project (reverse creation order).
All management calls use the JWT bearer token.

In [ ]:
print(api_call(
    "DELETE", f"/projects/{PROJECT_ID}/collections/{COLLECTION_ID}"
).json())

print(api_call(
    "DELETE", f"/projects/{PROJECT_ID}/keys/delete_keys",
    params={"key_category": "all"},
).json())

print(api_call("DELETE", f"/projects/{PROJECT_ID}").json())
print("\ncleanup complete")

## API quick reference (as used here)

| Action | Method & path | Auth |
|---|---|---|
| Login | `POST /api/v1/token` (form) | — |
| Create project | `POST /api/v1/projects` | JWT |
| Create key | `POST /api/v1/projects/{pid}/keys` | JWT |
| Create collection | `POST /api/v1/projects/{pid}/collections` | master / JWT |
| Send data | `POST /api/v1/projects/{pid}/collections/{cid}/send_data` | write / master |
| Query data | `GET /api/v1/projects/{pid}/collections/{cid}/get_data` | read / write / master / JWT |
| Statistics | `GET /api/v1/projects/{pid}/collections/{cid}/statistics` | read / write / master / JWT |
| Delete data | `DELETE /api/v1/projects/{pid}/collections/{cid}/delete_data` | master / JWT |
| Delete collection | `DELETE /api/v1/projects/{pid}/collections/{cid}` | master / JWT |
| Delete keys | `DELETE /api/v1/projects/{pid}/keys/delete_keys?key_category=all` | JWT |
| Delete project | `DELETE /api/v1/projects/{pid}` | JWT |

Supported filter operators: `eq`, `ne`, `lt`, `lte`, `gt`, `gte`, `in`, `contains`,
`not_contains`, plus an `or` operator with nested `operands`.
